<a href="https://colab.research.google.com/github/shreyashan-git/ds2002-fa26/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT
    tracks.title,
    artists.name AS artist_name,
    artists.country
FROM tracks
JOIN artists
    ON tracks.artist_id = artists.artist_id;
''')

,title,artist_name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


This code joins the tracks and artists tables using their matching artist id. It returns each track’s title along with the artist’s name and country.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT
    genre,
    AVG(seconds) AS avg_seconds
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY avg_seconds DESC
LIMIT 1;
''')

,genre,avg_seconds
0,Electronic,287.5


This code basically calculates the average track length for each genre while excluding tracks with no genre. It then sorts the averages from longest to shortest and returns the genre with the longest average track length.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT
    user,
    COUNT(*) AS plays,
    COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
''')

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


This code basically counts the total number of plays and the number of different tracks played by each user. It then groups the results by user and lists the users in alphabetical order.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT
    tracks.track_id,
    tracks.title
FROM tracks
LEFT JOIN plays
    ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL;
''')

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


This code basically finds tracks that have never been played by keeping all tracks and matching them with plays. It then returns the track ID and title for tracks that have no matching play record.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT
    plays.user,
    SUM(tracks.seconds) AS total_seconds,
    ROUND(SUM(tracks.seconds) / 60.0, 1) AS total_minutes
FROM plays
JOIN tracks
    ON plays.track_id = tracks.track_id
GROUP BY plays.user
ORDER BY total_seconds DESC;
''')

,user,total_seconds,total_minutes
0,ava,885,14.8
1,ben,716,11.9
2,dan,510,8.5
3,cara,501,8.3


This code basically calculates the total listening time for each user by adding the lengths of all tracks they played. It then shows the time in seconds and minutes and orders users from most to least listening time.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT
    track_id,
    title
FROM tracks
WHERE genre != 'Pop'
   OR genre IS NULL;
''')

# WHERE genre != 'Pop' by itself would have excluded the untagged track because NULL is unknown, so it does not evaluate as TRUE for != 'Pop'

,track_id,title
0,12,Foothills
1,13,Aurora
2,14,Nightfall
3,15,Sol
4,16,Coastline
5,17,Ridgeline
6,18,Untitled Demo


This code basically finds all tracks that are not Pop or do not have a genre assigned. It then returns the track ID and title for those tracks.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT
    played_on,
    COUNT(*) AS play_count,
    COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC;
''')

,played_on,play_count,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


This code basically counts the number of plays and unique users for each date. It then groups the results by date and orders them from earliest to latest.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
q1= q('''
SELECT
    tracks.title,
    artists.name AS artist_name,
    artists.country
FROM tracks
JOIN artists
    ON tracks.artist_id = artists.artist_id;
''')
q4= q('''
SELECT
    tracks.track_id,
    tracks.title
FROM tracks
LEFT JOIN plays
    ON tracks.track_id = plays.track_id
WHERE plays.track_id IS NULL;
''')
q3 = q('''
SELECT
    user,
    COUNT(*) AS plays,
    COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
''')

assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I had Q3 wrong because I named the count column something else instead of plays. The final check expected a column named plays, so even though my counts were correct, the check failed with an error. It took me a while to figure it out but I fixed it by changing it to COUNT(*) AS plays.